In [1]:
from pyflink.table import EnvironmentSettings, TableEnvironment
from faker import Faker

# Create a batch environment
env_settings = EnvironmentSettings.in_batch_mode()
table_env = TableEnvironment.create(env_settings)

# Intialize Faker
fake = Faker()

# Generate data
data = [(fake.name(), fake.city(), fake.state()) for _ in range(10)]

# Define column names
column_names = ['name', 'city', 'state']

# Create table
table = table_env.from_elements(data, schema=column_names)

# Print table
table.execute().print()

/Users/codebase/Documents/codebase/courses/learn-flink/.venv/lib/python3.8/site-packages/apache_beam/__init__.py:79: UserWarning: Python 3.8 reaches EOL in October 2024 and support will be removed from Apache Beam in version 2.61.0. See https://github.com/apache/beam/issues/31192 for more information.
  warnings.warn(


+--------------------------------+--------------------------------+--------------------------------+
|                           name |                           city |                          state |
+--------------------------------+--------------------------------+--------------------------------+
|                 Curtis Chapman |                    Michaelbury |                      Tennessee |
|                      John Reed |                New Lindseytown |                    Mississippi |
|                   Nicole Smith |                     Harryville |                      Tennessee |
|                   Jordan Henry |                      Adamsview |                 South Carolina |
|                   Joseph Brown |                     East Tammy |                 South Carolina |
|                    Luis Wilson |                East Sandraberg |                  Massachusetts |
|                 Jason Castillo |                       New Ruth |                        

In [2]:
table.execute().collect().next()

<Row('Curtis Chapman', 'Michaelbury', 'Tennessee')>

In [3]:
table.execute().collect().next()

<Row('Curtis Chapman', 'Michaelbury', 'Tennessee')>

In [4]:
table_env.create_temporary_view('source_table', table)

In [5]:
table_env.execute_sql(
    """
    SELECT * FROM source_table
    WHERE city = 'Lopeztown'
    """
).print()

Empty set


### Selecting a column

In [6]:
from pyflink.table.expressions import col

table.select(col('name'), col('city')).execute().print()

+--------------------------------+--------------------------------+
|                           name |                           city |
+--------------------------------+--------------------------------+
|                 Curtis Chapman |                    Michaelbury |
|                      John Reed |                New Lindseytown |
|                   Nicole Smith |                     Harryville |
|                   Jordan Henry |                      Adamsview |
|                   Joseph Brown |                     East Tammy |
|                    Luis Wilson |                East Sandraberg |
|                 Jason Castillo |                       New Ruth |
|                   Robert White |                        Foxside |
|              Mr. Colton Guzman |                  East Ericbury |
|                    Dylan Hogan |                       New Seth |
+--------------------------------+--------------------------------+
10 rows in set


### Filtering the data

In [7]:
from pyflink.table.expressions import col

(
    table
        .filter(col('state') == 'California')
        .select(col('name'), col('city'), col('state'))
        .execute()
        .print()
)

Empty set


### GroupBy

In [8]:
from pyflink.table.expressions import col

(
    table
    .group_by(col('state'))
    .select(
        col('state').alias('state'), 
        col('name').count.alias('state_wise_count')
    )
    .execute()
    .print()
)

+--------------------------------+----------------------+
|                          state |     state_wise_count |
+--------------------------------+----------------------+
|                      Tennessee |                    2 |
|                 South Carolina |                    2 |
|                         Nevada |                    1 |
|                    Mississippi |                    1 |
|                         Kansas |                    1 |
|                           Iowa |                    1 |
|                          Texas |                    1 |
|                  Massachusetts |                    1 |
+--------------------------------+----------------------+
8 rows in set


### Creating a SINK

In [9]:
table_env.execute_sql(
    """
    CREATE TABLE print_sink (
        name STRING,
        city STRING,
        state STRING
    )
    WITH (
        'connector' = 'print'
        )
    """
)

In [10]:
table_env.execute_sql(
    """
    INSERT INTO print_sink
        SELECT * FROM source_table
    """
).wait()

1> +I[Curtis Chapman, Michaelbury, Tennessee]
1> +I[John Reed, New Lindseytown, Mississippi]
1> +I[Nicole Smith, Harryville, Tennessee]
1> +I[Jordan Henry, Adamsview, South Carolina]
1> +I[Joseph Brown, East Tammy, South Carolina]
1> +I[Luis Wilson, East Sandraberg, Massachusetts]
1> +I[Jason Castillo, New Ruth, Kansas]
1> +I[Robert White, Foxside, Nevada]
1> +I[Mr. Colton Guzman, East Ericbury, Iowa]
1> +I[Dylan Hogan, New Seth, Texas]


### Collect the results

In [12]:
table_result = table_env.execute_sql(
    """
    SELECT * FROM source_table
    """
)
table_result

In [20]:
with table_result.collect() as results:
    for r in results:
        print(r)

<Row('Curtis Chapman', 'Michaelbury', 'Tennessee')>
<Row('John Reed', 'New Lindseytown', 'Mississippi')>
<Row('Nicole Smith', 'Harryville', 'Tennessee')>
<Row('Jordan Henry', 'Adamsview', 'South Carolina')>
<Row('Joseph Brown', 'East Tammy', 'South Carolina')>
<Row('Luis Wilson', 'East Sandraberg', 'Massachusetts')>
<Row('Jason Castillo', 'New Ruth', 'Kansas')>
<Row('Robert White', 'Foxside', 'Nevada')>
<Row('Mr. Colton Guzman', 'East Ericbury', 'Iowa')>
<Row('Dylan Hogan', 'New Seth', 'Texas')>


### Pandas Integration

In [24]:
# Table object to pandas dataframe
pdf = table.to_pandas()
pdf

,name,city,state
0,Curtis Chapman,Michaelbury,Tennessee
1,John Reed,New Lindseytown,Mississippi
2,Nicole Smith,Harryville,Tennessee
3,Jordan Henry,Adamsview,South Carolina
4,Joseph Brown,East Tammy,South Carolina
5,Luis Wilson,East Sandraberg,Massachusetts
6,Jason Castillo,New Ruth,Kansas
7,Robert White,Foxside,Nevada
8,Mr. Colton Guzman,East Ericbury,Iowa
9,Dylan Hogan,New Seth,Texas


In [27]:
# pandas to table object
table_temp = table_env.from_pandas(pdf)
table_temp.execute().print()

+--------------------------------+--------------------------------+--------------------------------+
|                           name |                           city |                          state |
+--------------------------------+--------------------------------+--------------------------------+
|                 Curtis Chapman |                    Michaelbury |                      Tennessee |
|                      John Reed |                New Lindseytown |                    Mississippi |
|                   Nicole Smith |                     Harryville |                      Tennessee |
|                   Jordan Henry |                      Adamsview |                 South Carolina |
|                   Joseph Brown |                     East Tammy |                 South Carolina |
|                    Luis Wilson |                East Sandraberg |                  Massachusetts |
|                 Jason Castillo |                       New Ruth |                        

### UDF

In [32]:
import uuid
import functools

from pyflink.table.udf import udf
from pyflink.table.expressions import call, col
from pyflink.table import TableEnvironment, EnvironmentSettings


def generate_uuid():
    return str(uuid.uuid4())

myhash = udf(functools.partial(generate_uuid), result_type='STRING')

result_table = table.select(col('city'), col('state'), call(myhash).alias('guuid'))

result_table.execute().print()

+--------------------------------+--------------------------------+--------------------------------+
|                           city |                          state |                          guuid |
+--------------------------------+--------------------------------+--------------------------------+
|                    Michaelbury |                      Tennessee | dc4edcc6-b974-4164-90f1-1a6... |
|                New Lindseytown |                    Mississippi | ac244552-dd8f-4615-9f0c-015... |
|                     Harryville |                      Tennessee | 57202ecf-a953-4eeb-83f5-820... |
|                      Adamsview |                 South Carolina | 2ce50b66-7acc-430a-a2f0-a87... |
|                     East Tammy |                 South Carolina | d3b3760e-7a0e-4ebf-b4a9-c5e... |
|                East Sandraberg |                  Massachusetts | babf0c53-975f-4ce6-9250-dcc... |
|                       New Ruth |                         Kansas | ba623445-be2f-490e-9450

In [28]:
a

NameError: name 'a' is not defined